# 05 - LLM Zero-Shot Baseline

Test Qwen-based zero-shot classification using Groq for Bangla toxic comment detection. The model is selected in `../configs/config.yaml`.

Strategy: batch 25 comments per request to stay within free tier limits (~1000 req/day on Groq).

In [ ]:
# Google Colab setup (does nothing when run locally).
# Clones the repo into Google Drive so results/ survive disconnects, then installs dependencies.
# Before the first run: Runtime > Change runtime type > T4 GPU, and upload train.csv (see below).
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/drive/MyDrive/Set_A'
    if not os.path.isdir(REPO_DIR):
        !git clone -b bangal_profanity https://github.com/mehediparvez/Set_A.git {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}/BanglaProfanityDetection/notebooks
    !pip install -q transformers datasets accelerate scikit-learn better-profanity groq google-generativeai pyyaml seaborn tabulate
    if not os.path.exists('../data/train.csv'):
        print(f'Missing dataset: upload train.csv to {REPO_DIR}/BanglaProfanityDetection/data/train.csv')

In [ ]:
import os
import time
import sys
sys.path.append('..')
from src.dataset import load_dataset
from src.evaluate import get_classification_report
from src.llm_baseline import run_llm_baseline
from src.results import build_evaluation, save_evaluation
from src.utils import load_config, ensure_dirs

config = load_config('../configs/config.yaml')
ensure_dirs('../results')

## Setup API Key

Set your Groq API key (free at https://console.groq.com):

In [ ]:
# Option 1: Colab Secrets (key icon in the left sidebar, name GROQ_API_KEY, Notebook access on)
if 'google.colab' in sys.modules and not os.environ.get('GROQ_API_KEY'):
    from google.colab import userdata
    try:
        os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    except Exception as e:
        print(f'Could not read the GROQ_API_KEY secret: {e}')

# Option 2: Set directly (don't commit your key)
# os.environ['GROQ_API_KEY'] = 'your-key-here'

assert os.environ.get('GROQ_API_KEY'), 'Set GROQ_API_KEY environment variable'

from groq import Groq
client = Groq()
LLM_CFG = config['llm']['groq']
LLM_MODEL = LLM_CFG['model']
BATCH_SIZE = LLM_CFG['batch_size']
# Check availability before starting the batch loop.
available_models = {model.id for model in client.models.list().data}
if LLM_MODEL not in available_models:
    raise ValueError(
        f'Model {LLM_MODEL!r} is unavailable. Update llm.groq.model in configs/config.yaml. '
        f'Available models: {sorted(available_models)}'
    )
print(f'Using model: {LLM_MODEL}, batch size: {BATCH_SIZE}')

# Smoke test on 3 comments before the full run, so a broken setup fails fast with a clear message
from src.llm_baseline import classify_batch_groq
_sample = classify_batch_groq(
    ['ভাই, ভিডিওটা অনেক সুন্দর হয়েছে', 'তুই একটা বোকা, চুপ থাক', 'ধন্যবাদ সবাইকে'],
    client, LLM_MODEL, LLM_CFG.get('reasoning_effort'),
)
assert _sample is not None, 'The model reply could not be parsed; see the printed reply above.'
print(f'Smoke test labels (expect roughly [0, 1, 0]): {_sample}')

## Load Test Data

In [ ]:
DATA_PATH = '../' + config['data']['train_path']

# Same stratified split as the fine-tuning notebooks, so every model sees the same test set
_, texts, _, labels = load_dataset(
    DATA_PATH,
    text_column=config['data']['text_column'],
    label_column=config['data']['label_column'],
    test_size=config['data']['test_size'],
    random_seed=config['data']['random_seed'],
)
print(f'Loaded {len(texts)} test samples')

## Batch Classification

`run_llm_baseline` sends batches of numbered comments, parses the numbered `toxic` / `non-toxic` reply,
and retries a batch when the reply can't be parsed. Batches that still fail are excluded from scoring
rather than counted as non-toxic.

In [ ]:
start_time = time.time()
output = run_llm_baseline(
    texts,
    labels,
    client,
    model_name=LLM_MODEL,
    batch_size=BATCH_SIZE,
    delay=LLM_CFG.get('delay_between_batches', 2),
    max_retries=LLM_CFG.get('max_retries', 3),
    provider='groq',
    reasoning_effort=LLM_CFG.get('reasoning_effort'),
)
inference_time = time.time() - start_time

print(f'\nTotal time: {inference_time:.1f}s')
print(f"Classified {output['successful']} / {output['total']} "
      f"({len(output['failed_indices'])} in failed batches)")
print(f"Predicted toxic: {sum(output['valid_predictions'])} / {output['successful']}")

## Evaluation

In [ ]:
y_true, y_pred = output['valid_labels'], output['valid_predictions']
evaluation = build_evaluation(
    y_true, y_pred, inference_time,
    extra={
        'provider': 'groq',
        'model': LLM_MODEL,
        'total': output['total'],
        'successful': output['successful'],
        'failed_indices': output['failed_indices'],
    },
)

print('=== LLM Baseline Results ===')
for k, v in evaluation['metrics'].items():
    print(f'  {k}: {v:.4f}')

print('\n=== Classification Report ===')
print(get_classification_report(y_true, y_pred))

print('\n=== Confusion Matrix ===')
print(evaluation['confusion_matrix'])

path = save_evaluation(evaluation, '../results', 'llm')
print(f'\nResults saved to {path}')

In [ ]:
# Colab: make sure everything written to Google Drive is actually uploaded before the runtime ends
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.flush_and_unmount()
    print('All results saved to Google Drive.')